# 🏥 تدريب نموذج كشف الأمراض الجلدية فائق الدقة (EfficientNet-B0)
### AI Skin Disease Detection via Compound Scaled CNN (EfficientNet-B0 & Balanced Classes)
### مصمم للتشغيل على Google Colab أو Kaggle GPU مع التصدير المباشر لـ TensorFlow.js للهاتف

---
### 🌟 لماذا تم اختيار معمارية EfficientNet-B0؟
1. **المقياس المتوازن (Compound Scaling):** توازن علمي بين عمق الشبكة وعرضها ودقة الإدخال، مما يعطي تفوقاً سريرياً في التقاط حدود الآفات والتصبغات الجلدية.
2. **أعلى دقة مقابل الحجم:** تحقق دقة تفوق شبكات مثل ResNet-50 مع حجم صغير جداً يناسب هواتف المحمول (~15MB).
3. **فئة الجلد الطبيعي/الحميد (Normal / Benign):** منع التشخيص العشوائي الخاطئ للبشرة السليمة.
4. **محاكاة كاميرا الهواتف (Data Augmentation):** مقاومة ظروف الإضاءة المنزلية وزوايا التصوير اليدوية.
5. **تصدير سلس لـ TensorFlow.js:** لإنتاج `model.json` وملفات الأوزان للدمج المباشر في React Native.

## ⚙️ الخطوة 1: فحص بيئة العمل وتأكيد تفعيل كارت الشاشة (GPU Verification)
> **تنبيه:** تأكد من اختيار **T4 GPU** في Google Colab من قائمة `Runtime` ➔ `Change runtime type`.

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print(f"🔹 إصدار TensorFlow: {tf.__version__}")

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"✅ تم تفعيل كارت الشاشة بنجاح: {len(gpus)} GPU(s)")
    for gpu in gpus:
        print(f"   🚀 {gpu.name}")
else:
    print("⚠️ تنبيه: لم يتم تفعيل GPU. يرجى تفعيله من إعدادات الجلسة لتسريع التدريب بأكثر من 10 أضعاف!")

## 📦 الخطوة 2: تثبيت أداة التحويل للهواتف (TensorFlow.js Converter)

In [ ]:
!pip install -q tensorflowjs

## 🎯 الخطوة 3: تحديد المعلمات والفئات الطبية (Hyperparameters & Classes)
المقاس الافتراضي القياسي لـ **EfficientNet-B0** هو **224x224** بكسل.

In [ ]:
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42

# قائمة الفئات الطبية الشاملة مع فئة الحماية (Normal_Skin)
CLASS_NAMES = [
    'Acne',          # حب الشباب
    'Carcinoma',     # أورام سرطانية قاعدية/حرشفية
    'Eczema',        # التهاب الجلد والأكزيما
    'Keratosis',     # التقران الدهني أو الشمسي
    'Milia',         # أكياس الدخينات الميليا
    'Rosacea',       # الوردية وتمدد الأوعية الوجهية
    'Normal_Skin'    # جلد طبيعي / سليم لمنع الإنذارات الخاطئة
]

NUM_CLASSES = len(CLASS_NAMES)
print(f"عدد الفئات المعتمدة: {NUM_CLASSES}")
for i, cls in enumerate(CLASS_NAMES):
    print(f"  [{i}] {cls}")

## 📂 الخطوة 4: خط أنابيب معالجة البيانات (Dataset Pipeline)
يتم تحميل الصور عبر التدفق السريع مع دعم التخزين المؤقت (Caching و Prefetching) لاستغلال كامل سرعة المعالج وكارت الشاشة.

In [ ]:
DATASET_DIR = "./skin_dataset"

# في حال أردت تجربة الكود فوراً بدون تحميل خارجي، ننشئ بنية مجلدات وهمية/تجريبية سريعة:
if not os.path.exists(DATASET_DIR):
    print(f"جاري إنشاء بنية المجلدات في: {DATASET_DIR}...")
    for split in ['train', 'val']:
        for cls in CLASS_NAMES:
            os.makedirs(os.path.join(DATASET_DIR, split, cls), exist_ok=True)
            for k in range(5):
                dummy_img = np.random.randint(50, 220, (224, 224, 3), dtype=np.uint8)
                tf.keras.utils.save_img(os.path.join(DATASET_DIR, split, cls, f"sample_{k}.jpg"), dummy_img)
    print("✅ تم إعداد مسار البيانات بنجاح!")

TRAIN_DIR = os.path.join(DATASET_DIR, 'train')
VAL_DIR = os.path.join(DATASET_DIR, 'val')

train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical',
    shuffle=True,
    seed=SEED
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    VAL_DIR,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical',
    shuffle=False,
    seed=SEED
)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)
print("🚀 تدفق البيانات جاهز للتدريب السريع!")

## 🎨 الخطوة 5: زيادة البيانات المتقدمة لمحاكاة عدسات الهواتف (Advanced Data Augmentation)
توليد تنويعات في التدوير، التكبير، والتدرج اللوني والانعكاس لضمان عدم تأثر الموديل بزاوية يد المريض أو إضاءة الغرفة.

In [ ]:
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.25),
    layers.RandomZoom(0.2),
    layers.RandomContrast(0.2),
    layers.RandomBrightness(0.15),
], name="skin_augmentation")

print("✅ تم إعداد طبقات محاكاة الكاميرا (Data Augmentation) بنجاح.")

## 🧠 الخطوة 6: بناء معمارية النموذج بالتعلم بالنقل عبر EfficientNet-B0

### تفاصيل البناء:
1. مدخلات بصيغة `(224, 224, 3)`.
2. معمارية **EfficientNetB0** محملة بأوزان `ImageNet` الأصلية.
3. طبقة `GlobalAveragePooling2D` متبوعة بـ `BatchNormalization`.
4. نسبة `Dropout` مدروسة لمنع الـ Overfitting وحفظ الصور.
5. طبقة نهائية بعدد الفئات مع تفعيل `Softmax` لحساب احتمالات الأمراض.

In [ ]:
def create_efficientnet_skin_model(num_classes=NUM_CLASSES):
    inputs = layers.Input(shape=(224, 224, 3), name="input_image")
    
    # تطبيق التحسينات الصورية
    x = data_augmentation(inputs)
    
    # المعالجة المسبقة الخاصة بـ EfficientNetB0
    x = tf.keras.applications.efficientnet.preprocess_input(x)
    
    # تحميل الشبكة الأساسية بأوزان ImageNet
    base_model = tf.keras.applications.EfficientNetB0(
        include_top=False,
        weights='imagenet',
        input_tensor=x,
        pooling='avg'
    )
    
    # تجميد طبقات الشبكة في المرحلة الأولى
    base_model.trainable = False
    
    x = base_model.output
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.4)(x)               # حماية من فرط التخصيص
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.2)(x)
    
    outputs = layers.Dense(num_classes, activation='softmax', name="skin_disease_output")(x)
    
    model = keras.Model(inputs=inputs, outputs=outputs, name="EfficientNet_Skin_Classifier")
    return model, base_model

model, base_model = create_efficientnet_skin_model()
print(f"✅ تم بناء معمارية EfficientNet-B0 الطبية بنجاح! إجمالي الطبقات: {len(base_model.layers)}")
model.summary()

## 🚀 الخطوة 7: تدريب المرحلة الأولى (Feature Extraction)
تدريب الطبقات العلوية لتتعلم تفسير الأنماط المستخرجة من EfficientNet.

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=[
        'accuracy',
        tf.keras.metrics.AUC(name='auc'),
        tf.keras.metrics.Recall(name='recall'),
        tf.keras.metrics.Precision(name='precision')
    ]
)

callbacks = [
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1)
]

EPOCHS_PHASE_1 = 12
print("🚀 بدء المرحلة الأولى من تدريب EfficientNet-B0...")
history_p1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_PHASE_1,
    callbacks=callbacks
)

## 🔬 الخطوة 8: تدريب المرحلة الثانية والضبط الدقيق الطبي (Fine-Tuning)
فك تجميد أعلى طبقات من **EfficientNet-B0** لضبط أوزان المرشحات (Filters) على الأنماط المجهرية للجلد.

In [ ]:
base_model.trainable = True
# فك تجميد آخر 25 طبقة فقط لضمان الثبات والسرعة
for layer in base_model.layers[:-25]:
    layer.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc'), tf.keras.metrics.Recall(name='recall')]
)

EPOCHS_PHASE_2 = 15
print("🔬 بدء مرحلة الـ Fine-Tuning لنموذج EfficientNet-B0...")
history_p2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_PHASE_2,
    callbacks=callbacks
)
print("🎉 اكتمل تدريب وضبط النموذج بكفاءة متناهية!")

## 📊 الخطوة 9: التقييم السريري الدقيق ومصفوفة الالتباس (Confusion Matrix)

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

y_true = []
y_pred = []

for images, labels in val_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(np.argmax(labels.numpy(), axis=1))
    y_pred.extend(np.argmax(preds, axis=1))

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title('مصفوفة الالتباس الطبية لنموذج EfficientNet-B0', fontsize=14)
plt.xlabel('التشخيص المتوقع (Predicted)')
plt.ylabel('التشخيص الفعلي (Actual)')
plt.xticks(rotation=45)
plt.show()

print("📋 التقرير التشخيصي التفصيلي (Classification Report):")
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, zero_division=0))

## 📱 الخطوة 10: التصدير بصيغة TensorFlow.js وضغط الملفات لتطبيق React Native

In [ ]:
SAVED_MODEL_DIR = "./saved_efficientnet_model"
TFJS_OUTPUT_DIR = "./tfjs_efficientnet_model"

os.makedirs(SAVED_MODEL_DIR, exist_ok=True)
os.makedirs(TFJS_OUTPUT_DIR, exist_ok=True)

# 1. حفظ نموذج Keras بصيغة .keras الحديثة
keras_path = os.path.join(SAVED_MODEL_DIR, "efficientnet_skin_model.keras")
model.save(keras_path)
print(f"✅ تم حفظ نموذج EfficientNet-B0 في: {keras_path}")

# 2. التحويل لـ TensorFlow.js Layers Model المتوافق مع الهواتف
!tensorflowjs_converter --input_format=keras {keras_path} {TFJS_OUTPUT_DIR}

print("\n📁 ملفات الموديل المحولة للهاتف:")
for f in os.listdir(TFJS_OUTPUT_DIR):
    file_size_kb = os.path.getsize(os.path.join(TFJS_OUTPUT_DIR, f)) / 1024
    print(f"  📄 {f} ({file_size_kb:.1f} KB)")

# 3. ضغط الملفات في أرشيف ZIP للتنزيل الفوري
!zip -r efficientnet_skin_model_tfjs.zip {TFJS_OUTPUT_DIR}
print("\n🎉 تم إنشاء الملف المضغوط: efficientnet_skin_model_tfjs.zip")
print("👉 قم بتنزيله واستبدال الملفات في assets/model/ داخل تطبيقنا!")